# Script Universal de Analise de Dados (Pandas + Plotly)

## Carregando Bibliotecas

In [167]:
import pandas as pd
import plotly.express as px

## Carregando Base de Dados (Pandas)

In [168]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url)
# Para Excel, use: pd.read_excel('seu_arquivo.xlsx')


## Dignóstico Rápido

In [169]:
print("--- PRIMEIRAS LINHAS ---")
display(df.head())

print("\n--- INFORMAÇÕES E TIPOS DE DADOS ---")
print(df.info())

print("\n--- RESUMO ESTATÍSTICO ---")
display(df.describe())


--- PRIMEIRAS LINHAS ---


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE



--- INFORMAÇÕES E TIPOS DE DADOS ---
<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.9 KB
None

--- RESUMO ESTATÍSTICO ---


,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
count,342.000000,342.000000,342.000000,342.000000
mean,43.921930,17.151170,200.915205,4201.754386
std,5.459584,1.974793,14.061714,801.954536
min,32.100000,13.100000,172.000000,2700.000000
25%,39.225000,15.600000,190.000000,3550.000000
50%,44.450000,17.300000,197.000000,4050.000000
75%,48.500000,18.700000,213.000000,4750.000000
max,59.600000,21.500000,231.000000,6300.000000


## Limpeza e Tratamento de Dados

In [170]:
if df.isnull().values.any():
    print("\n--- VALORES NULOS ---")
    display(df.isnull().sum())
    print("--- REMOVENDO LINHAS COM VALORES NULOS ---")
    df_cleaned = df.dropna()
    display(df_cleaned.describe())

if df_cleaned.duplicated().any():
    print("\n--- EXISTEM LINHAS DUPLICADAS ---")
    print(df_cleaned[df_cleaned.duplicated()])
    print("\n--- REMOVENDO LINHAS DUPLICADAS ---")
    df_cleaned = df_cleaned.drop_duplicates()
    display(df_cleaned.describe())



--- VALORES NULOS ---


species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

--- REMOVENDO LINHAS COM VALORES NULOS ---


,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
count,333.000000,333.000000,333.000000,333.000000
mean,43.992793,17.164865,200.966967,4207.057057
std,5.468668,1.969235,14.015765,805.215802
min,32.100000,13.100000,172.000000,2700.000000
25%,39.500000,15.600000,190.000000,3550.000000
50%,44.500000,17.300000,197.000000,4050.000000
75%,48.600000,18.700000,213.000000,4775.000000
max,59.600000,21.500000,231.000000,6300.000000


## Respondendo Perguntas do Dataframe

In [171]:
contagem = df.groupby('species')['body_mass_g'].count().reset_index()
media = df.groupby('species')['body_mass_g'].mean().reset_index()

resultado = pd.merge(contagem, media, on='species', suffixes=('_count', '_mean')) 
resultado.rename(columns={'body_mass_g_count': 'quantidade', 'body_mass_g_mean': 'media'}, inplace=True)
resultado['media'] = resultado['media'].round(2)
# print("\n--- MÉDIA DE PESO POR ESPÉCIE (sem limpeza) ---")
# display(resultado)

contagem = df_cleaned.groupby('species')['body_mass_g'].count().reset_index()
media = df_cleaned.groupby('species')['body_mass_g'].mean().reset_index()

resultado_cleaned = pd.merge(contagem, media, on='species', suffixes=('_count', '_mean')) 
resultado_cleaned.rename(columns={'body_mass_g_count': 'qtd_cleaned', 'body_mass_g_mean': 'media_cleaned'}, inplace=True)
resultado_cleaned['media_cleaned'] = resultado_cleaned['media_cleaned'].round(2)
# print("\n--- MÉDIA DE PESO POR ESPÉCIE (com limpeza) ---")
# display(resultado_cleaned)

joint = pd.merge(resultado, resultado_cleaned, on='species', suffixes=('', '_cleaned'))
display(joint)

,species,quantidade,media,qtd_cleaned,media_cleaned
0,Adelie,151,3700.66,146,3706.16
1,Chinstrap,68,3733.09,68,3733.09
2,Gentoo,123,5076.02,119,5092.44


In [172]:
representacao = resultado_cleaned['qtd_cleaned'] / resultado_cleaned['qtd_cleaned'].sum() * 100
representacao = representacao.round(2)
representacao_df = pd.DataFrame({'species': resultado_cleaned['species'], 'representacao_%': representacao})

joint_representacao = pd.merge(resultado_cleaned, representacao_df, on='species')
display(joint_representacao)

,species,qtd_cleaned,media_cleaned,representacao_%
0,Adelie,146,3706.16,43.84
1,Chinstrap,68,3733.09,20.42
2,Gentoo,119,5092.44,35.74


In [173]:
# Describe agrupado por species
# Uma tabela por indicador
for column in df_cleaned.select_dtypes(include=['float64', 'int64']).columns:
    desc_column = df_cleaned.groupby('species')[column].describe()
    print(f"\n--- DESCRIÇÃO DE {column.upper()} POR ESPÉCIE ---")
    display(desc_column.round(2).transpose())
    


--- DESCRIÇÃO DE BILL_LENGTH_MM POR ESPÉCIE ---


species,Adelie,Chinstrap,Gentoo
count,146.00,68.00,119.00
mean,38.82,48.83,47.57
std,2.66,3.34,3.11
min,32.10,40.90,40.90
25%,36.72,46.35,45.35
50%,38.85,49.55,47.40
75%,40.78,51.08,49.60
max,46.00,58.00,59.60



--- DESCRIÇÃO DE BILL_DEPTH_MM POR ESPÉCIE ---


species,Adelie,Chinstrap,Gentoo
count,146.00,68.00,119.00
mean,18.35,18.42,15.00
std,1.22,1.14,0.99
min,15.50,16.40,13.10
25%,17.50,17.50,14.20
50%,18.40,18.45,15.00
75%,19.00,19.40,15.75
max,21.50,20.80,17.30



--- DESCRIÇÃO DE FLIPPER_LENGTH_MM POR ESPÉCIE ---


species,Adelie,Chinstrap,Gentoo
count,146.00,68.00,119.00
mean,190.10,195.82,217.24
std,6.52,7.13,6.59
min,172.00,178.00,203.00
25%,186.00,191.00,212.00
50%,190.00,196.00,216.00
75%,195.00,201.00,221.50
max,210.00,212.00,231.00



--- DESCRIÇÃO DE BODY_MASS_G POR ESPÉCIE ---


species,Adelie,Chinstrap,Gentoo
count,146.00,68.00,119.00
mean,3706.16,3733.09,5092.44
std,458.62,384.34,501.48
min,2850.00,2700.00,3950.00
25%,3362.50,3487.50,4700.00
50%,3700.00,3700.00,5050.00
75%,4000.00,3950.00,5500.00
max,4775.00,4800.00,6300.00


## Representacoes Graficas Rapidas

In [174]:
# Grafico de barras comparativo e Iterativo
fig = px.bar(
    resultado_cleaned, 
    x='species', 
    y='media_cleaned', 
    color='species',
    title='<b>Comparativo Prático de Dados</b>',
    template='plotly_white'
)
fig.show()

## Gráficos mais Avançados

In [ ]:
#Colunas específicas para análise dos dados dos pinguins
caracteristicas = [
    'bill_length_mm',
    'bill_depth_mm',
    'flipper_length_mm',
    'body_mass_g'
]

caracteristicas_pt = [
    'Comp do Bico (mm)',
    'Prof do Bico (mm)',
    'Comp da Nadadeira (mm)',
    'Massa Corporal (g)'
]

df_cleaned.rename(columns=dict(zip(caracteristicas, caracteristicas_pt)), inplace=True)

### Grafico de Dispersão Cruzado (Scatter Plot Matrix)

In [ ]:
# Matriz de dispersão cruzada das características numéricas por espécie
fig_scatter = px.scatter_matrix(
    df_cleaned,
    dimensions=caracteristicas_pt,
    color='species',
    title='<b>Relação entre as características dos pinguins por espécie</b>',
    template='plotly_white'
)

fig_scatter.update_traces(diagonal_visible=False)
fig_scatter.update_layout(
    width=1200,
    height=900,
    legend_title_text='Espécie'
)

fig_scatter.show()

### Boxplot

In [178]:
numeric_columns = df_cleaned.select_dtypes(include=['float64', 'int64']).columns
for column in numeric_columns:
    fig = px.box(
        df_cleaned, 
        x='species', 
        y=column, 
        color='species',
        title=f'<b>Distribuição de {column} por Espécie</b>',
        template='plotly_white'
    )
    fig.update_layout(
        xaxis_title='Espécie',
        yaxis_title=column,
        legend_title='Espécie'
    )
    fig.show()